In [28]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder,StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.svm import SVC
from tqdm import tqdm
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.pipeline import Pipeline

In [29]:
image=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Image_Segmentation\\Image_Segmentation.csv")
image

,Class,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean
0,BRICKFACE,188,133,9,0.000000,0.0,0.333333,0.266667,0.500000,0.077778,6.666666,8.333334,7.777778,3.888889,5.000000,3.333333,-8.333333,8.444445,0.538580,-0.924817
1,BRICKFACE,105,139,9,0.000000,0.0,0.277778,0.107407,0.833333,0.522222,6.111111,7.555555,7.222222,3.555556,4.333334,3.333333,-7.666666,7.555555,0.532628,-0.965946
2,BRICKFACE,34,137,9,0.000000,0.0,0.500000,0.166667,1.111111,0.474074,5.851852,7.777778,6.444445,3.333333,5.777778,1.777778,-7.555555,7.777778,0.573633,-0.744272
3,BRICKFACE,39,111,9,0.000000,0.0,0.722222,0.374074,0.888889,0.429629,6.037037,7.000000,7.666666,3.444444,2.888889,4.888889,-7.777778,7.888889,0.562919,-1.175773
4,BRICKFACE,16,128,9,0.000000,0.0,0.500000,0.077778,0.666667,0.311111,5.555555,6.888889,6.666666,3.111111,4.000000,3.333333,-7.333334,7.111111,0.561508,-0.985811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,GRASS,36,243,9,0.111111,0.0,1.888889,1.851851,2.000000,0.711110,13.333333,9.888889,12.111111,18.000000,-10.333333,-3.666667,14.000000,18.000000,0.452229,2.368311
205,GRASS,186,218,9,0.000000,0.0,1.166667,0.744444,1.166667,0.655555,13.703704,10.666667,12.666667,17.777779,-9.111111,-3.111111,12.222222,17.777779,0.401347,2.382684
206,GRASS,197,236,9,0.000000,0.0,2.444444,6.829628,3.333333,7.599998,16.074074,13.111111,16.666668,18.444445,-8.888889,1.777778,7.111111,18.555555,0.292729,2.789800
207,GRASS,208,240,9,0.111111,0.0,1.055556,0.862963,2.444444,5.007407,14.148149,10.888889,13.000000,18.555555,-9.777778,-3.444444,13.222222,18.555555,0.421621,2.392487


In [30]:
le=LabelEncoder()
image['Class']=le.fit_transform(image['Class'])
X,y=image.drop('Class',axis=1), image['Class']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

In [31]:
scaler=StandardScaler().set_output(transform="pandas")
X_trn_scl=scaler.fit_transform(X_train)#mean and SD
X_tst_scl=scaler.transform(X_test)

In [32]:
Cs=np.linspace(0.001,5,20)#hyperparameter
dfs=["ovo","ovr"]# multiclass stratergies
scores=[]
for c in tqdm(Cs):
    for f in dfs:
        svm=SVC(kernel='linear',C=c,probability=True, random_state=26,decision_function_shape=f)# training the model
        svm.fit(X_trn_scl,y_train)
        y_pred_prob=svm.predict_proba(X_tst_scl)
        scores.append([c,f,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['C','dfs','score'])
df_scores.sort_values('score',ascending=True)

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 29.29it/s]


,C,dfs,score
3,0.264105,ovr,0.629300
2,0.264105,ovo,0.629300
5,0.527211,ovr,0.661606
4,0.527211,ovo,0.661606
6,0.790316,ovo,0.679109
7,0.790316,ovr,0.679109
9,1.053421,ovr,0.688156
8,1.053421,ovo,0.688156
10,1.316526,ovo,0.693341
11,1.316526,ovr,0.693341


In [33]:
glass=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Glass_Identification\\Glass.csv")
glass

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0,building_windows_float_processed
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0,building_windows_float_processed
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0,building_windows_float_processed
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0,building_windows_float_processed
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0,building_windows_float_processed
...,...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0,headlamps
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0,headlamps
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0,headlamps
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0,headlamps


In [34]:
le=LabelEncoder()
glass['Type']=le.fit_transform(glass['Type'])
X,y=glass.drop('Type',axis=1), glass['Type']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

In [35]:
scaler=StandardScaler().set_output(transform="pandas")
X_trn_scl=scaler.fit_transform(X_train)#mean and SD
X_tst_scl=scaler.transform(X_test)

In [36]:
Cs=np.linspace(0.001,5,20)#hyperparameter
dfs=["ovo","ovr"]# multiclass stratergies
scores=[]
for c in tqdm(Cs):
    for f in dfs:
        svm=SVC(kernel='linear',C=c,probability=True, random_state=26,decision_function_shape=f)# training the model
        svm.fit(X_trn_scl,y_train)
        y_pred_prob=svm.predict_proba(X_tst_scl)
        scores.append([c,f,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['C','dfs','score'])
df_scores.sort_values('score',ascending=True)

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 26.32it/s]


,C,dfs,score
6,0.790316,ovo,0.951070
7,0.790316,ovr,0.951070
10,1.316526,ovo,0.953640
11,1.316526,ovr,0.953640
8,1.053421,ovo,0.953687
9,1.053421,ovr,0.953687
13,1.579632,ovr,0.954680
12,1.579632,ovo,0.954680
17,2.105842,ovr,0.955157
16,2.105842,ovo,0.955157


In [48]:
Cs=np.linspace(0.001,5,20)
Gs=np.linspace(0.001,5,20)
dfs=['ovo','ovr']
scores=[]
for c in Cs:
    for f in dfs:
        for g in Gs:
            svm=SVC(kernel='rbf',C=c,probability=True, random_state=26,gamma=g,decision_function_shape=f)
            svm.fit(X_trn_scl,y_train)
            y_pred_prob=svm.predict_proba(X_tst_scl)
            scores.append([c,f,g,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['C','g','dfs','score'])
df_scores.sort_values('score',ascending=True)

,C,g,dfs,score
241,1.579632,ovo,0.264105,0.491097
261,1.579632,ovr,0.264105,0.491097
301,1.842737,ovr,0.264105,0.492866
281,1.842737,ovo,0.264105,0.492866
221,1.316526,ovr,0.264105,0.493293
...,...,...,...,...
75,0.264105,ovr,3.947579,1.999678
73,0.264105,ovr,3.421368,1.999724
53,0.264105,ovo,3.421368,1.999724
54,0.264105,ovo,3.684474,2.000049


In [ ]:
ohe=